# Wealth Management & Advisor AI

- [Why Wealth Management Differs](#Why-Wealth-Management-Differs)
  - [Portfolio vs product recommendation](#Portfolio-vs-product-recommendation)
- [Suitability-Constrained Recommendation](#Suitability-Constrained-Recommendation)
  - [Filter, then rank](#Filter,-then-rank)
  - [Cold start for a new client](#Cold-start-for-a-new-client)
- [Advisor-Facing Ranking](#Advisor-Facing-Ranking)
  - [Next-best-conversation list for relationship managers](#Next-best-conversation-list-for-relationship-managers)
- [Rebalancing: Rules and ML](#Rebalancing:-Rules-and-ML)
  - [Drift thresholds vs cost-aware triggers](#Drift-thresholds-vs-cost-aware-triggers)
- [Offline Evaluation with Delayed, Noisy Outcomes](#Offline-Evaluation-with-Delayed,-Noisy-Outcomes)
  - [Measure what the system controls](#Measure-what-the-system-controls)
- [LLM Assistants for Advisors](#LLM-Assistants-for-Advisors)
  - [Grounding and product facts](#Grounding-and-product-facts)
- [Explainability and Compliance](#Explainability-and-Compliance)
  - [Reg BI, fiduciary duty and MiFID II suitability](#Reg-BI,-fiduciary-duty-and-MiFID-II-suitability)
- [Blogs](#Blogs)
  - [Pitfalls](#Pitfalls)

### Why Wealth Management Differs

A normal recommender optimizes engagement or revenue and learns from fast feedback. Wealth recommendations differ in four ways:

| | Retail recommender | Wealth recommender |
|---|---|---|
| Objective | clicks, purchases, engagement | the client's best interest: fit to goals and risk, cost, tax |
| Constraints | few | suitability rules, fiduciary duty, Reg BI (US) and MiFID II suitability (EU) |
| Feedback | fast and plentiful | slow and noisy (returns take years and are dominated by the market) |
| Users | the customer | the client and the advisor or relationship manager (RM) |

Two consequences run through everything below: constraints are applied as filters before ranking, and success is measured on things the system controls (fit, follow-through), not on investment returns.

#### Portfolio vs product recommendation

A portfolio-level recommendation asks "is this client's overall allocation right for their goals, horizon and risk?" and acts on the gap to a target (rebalance, add a missing asset class, reduce concentration). A product-level recommendation asks "which fund or instrument fills this role?" and compares cost, tracking, tax treatment and liquidity within an asset class. The objective changes accordingly: portfolio fit and diversification for the first, net-of-cost suitability within a role for the second. Doing product ranking before fixing the allocation recommends good products for the wrong portfolio.

### Suitability-Constrained Recommendation

#### Filter, then rank

Suitability is a hard constraint, not a feature the model can trade off. Remove every product that violates the client's risk capacity, horizon or liquidity needs first, then rank what remains by net benefit (fit improvement minus cost). Every recommendation then carries its reasons, which the compliance record needs anyway.

Synthetic clients and products:

In [1]:
import numpy as np, pandas as pd

products = pd.DataFrame({
    "product": ["Global equity ETF", "Aggregate bond ETF", "Short-term treasury fund", "Emerging markets fund", "Structured note"],
    "risk": [4, 2, 1, 5, 5], "fee_pct": [0.10, 0.08, 0.05, 0.55, 1.20], "liquid": [True, True, True, True, False],
    "role": ["growth", "income", "stability", "growth", "growth"]})
client = {"max_product_risk": 4, "needs_liquidity": True,              # suitability limits from the client's profile
          "target": {"growth": 0.50, "income": 0.30, "stability": 0.20},
          "current": {"growth": 0.35, "income": 0.30, "stability": 0.35}}

ok = products[(products.risk <= client["max_product_risk"]) & (products.liquid | (not client["needs_liquidity"]))].copy()
gap = {r: client["target"][r] - client["current"][r] for r in client["target"]}     # positive = underweight
ok["fit_gain"] = ok.role.map(gap)
ok["net_benefit"] = ok.fit_gain - ok.fee_pct / 100 * 10                # fit improvement minus a cost penalty
ok["reason"] = ok.apply(lambda r: f"{r.role} is {gap[r.role]:+.0%} vs target; fee {r.fee_pct:.2f}%", axis=1)
print("removed by suitability:", sorted(set(products["product"]) - set(ok["product"])))
print(ok.sort_values("net_benefit", ascending=False)[["product", "net_benefit", "reason"]].round(3).to_string(index=False))

removed by suitability: ['Emerging markets fund', 'Structured note']
                 product  net_benefit                                 reason
       Global equity ETF        0.140    growth is +15% vs target; fee 0.10%
      Aggregate bond ETF       -0.008     income is +0% vs target; fee 0.08%
Short-term treasury fund       -0.155 stability is -15% vs target; fee 0.05%


The structured note and the emerging-markets fund never reach the ranking because they fail suitability or liquidity. Among the rest, the equity ETF ranks first because the client is 15 points underweight in growth and its fee is low, and the reason string explains it in the client's own terms.

#### Cold start for a new client

A new client has no portfolio and no behavior, so the popular-items fallback is not available and would not be suitable anyway. The standard start is a risk questionnaire (risk capacity, horizon, goals) that maps to a model portfolio, which acts as the prior. As real holdings, transactions and feedback accumulate, the recommendation shifts weight from the model portfolio toward what the client's own data says. Betterment, for example, publishes individualized risk levels (101 of them) derived from the client's inputs.

In [2]:
def model_portfolio(risk_score):                                      # risk_score 1 (low) to 5 (high)
    stocks = 0.20 + 0.15 * (risk_score - 1)
    return {"stocks": round(stocks, 2), "bonds": round(1 - stocks, 2)}

for score in (1, 3, 5): print("risk score", score, "->", model_portfolio(score))

prior = np.array(list(model_portfolio(3).values()))                   # questionnaire prior: 50/50
observed = np.array([0.72, 0.28])                                     # what the client actually holds elsewhere
for months in (0, 3, 12, 24):
    w = months / (months + 12)                                        # trust in observed behavior grows with data
    print(f"after {months:2d} months: blended stocks weight {(1 - w) * prior[0] + w * observed[0]:.2f}")

risk score 1 -> {'stocks': 0.2, 'bonds': 0.8}
risk score 3 -> {'stocks': 0.5, 'bonds': 0.5}
risk score 5 -> {'stocks': 0.8, 'bonds': 0.2}
after  0 months: blended stocks weight 0.50
after  3 months: blended stocks weight 0.54
after 12 months: blended stocks weight 0.61
after 24 months: blended stocks weight 0.65


### Advisor-Facing Ranking

#### Next-best-conversation list for relationship managers

When the user is the advisor, the task is to rank which clients to contact this week, within the RM's capacity. A workable score combines the likelihood the client responds, the value at stake (assets, gap to target, a life event), and urgency (idle cash, large drift, a client at risk of leaving), with each lead carrying a reason. The output is a capped list, because an RM can hold only so many conversations, and its quality is judged by the follow-through on the top of the list.

Synthetic book of 150 clients:

In [3]:
rng = np.random.default_rng(0)
n = 150
book = pd.DataFrame({"client": [f"C{i:03d}" for i in range(n)],
                     "aum": rng.lognormal(12.5, 0.8, n),
                     "drift": rng.beta(1.5, 8, n),                    # gap to target allocation
                     "idle_cash_share": rng.beta(1, 6, n),
                     "days_since_contact": rng.integers(5, 240, n),
                     "attrition_risk": rng.beta(1.2, 8, n)})
book["p_respond"] = np.clip(0.15 + 0.5 * book.drift + 0.3 * book.idle_cash_share, 0, 1)
book["value"] = book.aum * (book.drift + book.idle_cash_share + book.attrition_risk) * 0.01
book["score"] = book.p_respond * book.value
top = book.sort_values("score", ascending=False).head(10)
def why(r):
    reasons = []
    if r.drift > 0.25: reasons.append(f"drift {r.drift:.0%}")
    if r.idle_cash_share > 0.25: reasons.append(f"idle cash {r.idle_cash_share:.0%}")
    if r.attrition_risk > 0.25: reasons.append("attrition risk")
    if r.days_since_contact > 150: reasons.append(f"{r.days_since_contact} days since contact")
    return ", ".join(reasons) or "high assets"
top["reasons"] = top.apply(why, axis=1)
print(top[["client", "aum", "score", "reasons"]].round(0).to_string(index=False))

client       aum  score                                           reasons
  C137  673425.0 2843.0  drift 37%, idle cash 40%, 239 days since contact
  C136  823906.0 2179.0 drift 37%, attrition risk, 182 days since contact
  C074 1152690.0 2077.0            attrition risk, 160 days since contact
  C038  502414.0 1806.0                     idle cash 42%, attrition risk
  C006  761618.0 1708.0                                         drift 32%
  C057  468286.0 1660.0  drift 41%, idle cash 34%, 235 days since contact
  C042  787564.0 1638.0             idle cash 30%, 239 days since contact
  C132  357517.0 1624.0                     idle cash 57%, attrition risk
  C046  861488.0 1585.0                                    attrition risk
  C135  617523.0 1540.0                                         drift 34%


Segmentation supports this: clustering clients on behavior (trading frequency, product mix, channel use) gives segments with different talking points, so the list suggests not only whom to call but what to talk about.

### Rebalancing: Rules and ML

#### Drift thresholds vs cost-aware triggers

A drift rule ("review when any asset class is 5 points from target") is simple to explain to a client or a regulator. The trade-off is between tracking the target closely, which needs frequent trades, and paying trading cost and tax on each rebalance. A tighter threshold trades more often, a looser one lets the portfolio wander. The simulation below makes that trade-off visible:

In [4]:
rng = np.random.default_rng(1)
days, trade_cost = 252 * 5, 0.0005                                    # five years of daily returns, 5 bps cost on traded value
r_stock = rng.normal(0.0004, 0.011, days); r_bond = rng.normal(0.0001, 0.003, days)

def run(threshold):
    w, trades, cost, gap = 0.60, 0, 0.0, []
    for rs, rb in zip(r_stock, r_bond):
        w = w * (1 + rs) / (w * (1 + rs) + (1 - w) * (1 + rb))        # weight drifts with returns
        gap.append(abs(w - 0.60))
        if abs(w - 0.60) > threshold:
            cost += abs(w - 0.60) * trade_cost; trades += 1; w = 0.60
    return trades, np.mean(gap), cost
print("threshold | rebalances | avg drift | trading cost (share of portfolio)")
for th in (0.01, 0.03, 0.05, 0.10):
    tr, g, c = run(th); print(f"   {th:4.0%}   | {tr:10d} | {g:8.2%}  | {c:.5f}")

threshold | rebalances | avg drift | trading cost (share of portfolio)
     1%   |         60 |    0.41%  | 0.00036
     3%   |          8 |    1.01%  | 0.00013
     5%   |          2 |    1.82%  | 0.00005
    10%   |          1 |    2.90%  | 0.00005


Tighter thresholds rebalance more often and keep the portfolio close to target, while looser ones save cost and let the allocation drift. An ML-driven rebalancer can add value by timing trades around cash flows, tax lots and costs, but it must still be explainable: a common compromise keeps the drift rule as the visible trigger and uses the model only to choose how and when within the allowed window.

### Offline Evaluation with Delayed, Noisy Outcomes

#### Measure what the system controls

Investment returns arrive years later and are dominated by the market, so they cannot judge a recommender. Use proxies that the recommendation actually moves: whether the advisor acts on the suggestion (acceptance and follow-through), how far the portfolio moved toward its target after an accepted suggestion (fit improvement), and downstream effects such as attrition, with a randomized holdout (some clients or RMs get the old process) as the clean comparison. For a new ranking, replay it on logged data with inverse propensity scoring if the old process randomized any decisions.

Comparing two ranking policies on simulated outcomes (follow-through and fit improvement):

In [5]:
rng = np.random.default_rng(2)
n = 5000
drift = rng.beta(1.5, 8, n); engaged = rng.random(n)
acts = rng.random(n) < (0.10 + 0.5 * engaged)                         # follow-through depends on engagement, not on drift
fit_gain = drift * acts                                               # portfolio moves toward target only if acted on
capacity = 500
policies = {"rank by propensity": engaged, "rank by gap to target": drift, "rank by gap x propensity": drift * engaged, "random": rng.random(n)}
for name, s in policies.items():
    top = np.argsort(-s)[:capacity]
    print(f"{name:26s} | follow-through {acts[top].mean():.2f} | total fit improvement {fit_gain[top].sum():6.1f}")

rank by propensity         | follow-through 0.55 | total fit improvement   42.2
rank by gap to target      | follow-through 0.34 | total fit improvement   68.0
rank by gap x propensity   | follow-through 0.48 | total fit improvement   80.8
random                     | follow-through 0.34 | total fit improvement   26.2


Ranking by propensity alone maximizes follow-through but pushes effort to engaged clients who may not need help. Ranking by gap alone finds the clients who need it most but many will not act. The combined score balances both, which is why the advisor-facing score above multiplies a response probability by the value at stake.

### LLM Assistants for Advisors

#### Grounding and product facts

An assistant that answers advisor questions (research, product details, policy) is a RAG system with a strict accuracy requirement. A classic failure: asked whether two ETFs are the same, the model says "identical, both track the same index" when they differ in weighting methodology, fees, domicile, replication method or currency hedging. The fix is to answer from structured product facts (a data source of record), compose comparisons from those fields, cite them, and decline when a field is missing, instead of letting the model fill gaps from memory. Evaluate every use case on a golden set reviewed by domain experts before release.

In [6]:
etf_facts = {
    "ETF-A": {"index": "S&P 500", "weighting": "market-cap", "fee_pct": 0.03, "domicile": "US", "replication": "full"},
    "ETF-B": {"index": "S&P 500", "weighting": "equal-weight", "fee_pct": 0.20, "domicile": "US", "replication": "full"}}

def compare(a, b, facts):
    fa, fb = facts[a], facts[b]
    diffs = {k: (fa[k], fb[k]) for k in fa if fa[k] != fb[k]}
    return diffs

claim = "identical"                                                    # what the model said
diffs = compare("ETF-A", "ETF-B", etf_facts)
print("claim:", claim, "| supported by the facts?", len(diffs) == 0)
print("differences to show the advisor:", diffs)

claim: identical | supported by the facts? False
differences to show the advisor: {'weighting': ('market-cap', 'equal-weight'), 'fee_pct': (0.03, 0.2)}


### Explainability and Compliance

#### Reg BI, fiduciary duty and MiFID II suitability

- **Reg BI (US broker-dealers)** requires acting in the retail customer's best interest, with disclosure, care, conflict-of-interest and compliance obligations. **Fiduciary duty** applies to investment advisers. **MiFID II (EU)** requires a suitability assessment and a suitability report to the client.
- Practical implications for a recommender: the objective cannot be firm revenue or engagement, product steering toward higher-fee products must be tested for explicitly, every recommendation needs recorded inputs, reasons and version, and a human advisor remains accountable.
- Reasons should be stated in the client's terms (gap to target, cost, concentration), not as raw model attributions.

Reason codes from the same quantities used for ranking:

In [7]:
factors = {"underweight in growth vs target": 0.15, "idle cash above policy": 0.08, "fee savings vs current fund": 0.02, "concentration in one holding": -0.03}
print("Recommendation: add to Global equity ETF")
for name, weight in sorted(factors.items(), key=lambda kv: -abs(kv[1])):
    print(f"  {'supports' if weight > 0 else 'argues against':14s} {name:38s} {weight:+.2f}")

Recommendation: add to Global equity ETF
  supports       underweight in growth vs target        +0.15
  supports       idle cash above policy                 +0.08
  argues against concentration in one holding           -0.03
  supports       fee savings vs current fund            +0.02


### Blogs

- **Morgan Stanley, Next Best Action.** Morgan Stanley describes a machine-learning system that suggests investment ideas and topics for each client and prompts advisors to engage them. MIT Sloan Management Review's account stresses that use is voluntary, that advisors stay central ("financial advising is a human-based game"), and that advisors using it generate relevant ideas faster and engage clients more. [The Pursuit of AI-Driven Wealth Management](https://sloanreview.mit.edu/article/the-pursuit-of-ai-driven-wealth-management/)
- **Morgan Stanley, AI assistant for advisors.** Morgan Stanley and OpenAI describe a GPT-4 assistant that answers advisors' questions from the firm's internal research and documents using retrieval, evaluated against expert-reviewed use cases before each release. The scale and adoption figures in published summaries (for example 98% of advisor teams, 100,000 documents) come from secondary reports of the OpenAI case study. [Morgan Stanley uses AI evals to shape the future of financial services](https://openai.com/index/morgan-stanley/)
- **DBS, nudges for relationship managers.** DBS says it built seven types of nudges and a platform that prompts relationship managers to talk to clients about recommended topics, which it reports gave 16% growth in client engagement. Its data mart holds 15,000 customer data points feeding more than 100 AI and ML algorithms, serving over 3.5 million retail and wealth customers in Singapore. [DBS: AI-powered personalised nudges and investment ideas](https://www.dbs.com/artificial-intelligence-machine-learning/index.html)
- **Betterment, personalized portfolios.** Betterment publishes 101 individualized risk levels (instead of three tiers) built with modern portfolio theory and capital market assumptions, with tax-aware features such as municipal bonds in taxable accounts. This is the "questionnaire to model portfolio" cold start in practice. [Betterment's portfolio construction methodology](https://www.betterment.com/resources/portfolio-construction-methodology)

Common pattern: the model suggests and the advisor decides, success is measured on engagement and follow-through, and the system is tested before release with expert-reviewed evaluations.

#### Pitfalls

- Optimizing engagement or revenue instead of client fit, and recommending products before fixing the allocation.
- Treating suitability as a soft feature instead of a hard filter.
- Judging the system by investment returns, which are too delayed and noisy.
- Ranking clients by propensity alone, which sends effort to the engaged and away from those who need help.
- Letting an LLM answer product questions from memory instead of from structured facts.
- Explanations that are model attributions rather than reasons the client and the regulator can follow.